# Gymnasium 소개와 CartPole 데모

**예상 소요 시간:** 30분

Gymnasium 라이브러리를 사용해 CartPole 환경을 처음 실행해봅니다.

> **Google Colab 전용** - 이 노트북은 Colab에서 실행하세요.

## 목표
- Gymnasium 환경 생성 방법 이해
- CartPole-v1의 기본 구조 파악
- `reset()`과 `step()` API 사용법 익히기

## 0. 환경 설정

In [ ]:
# 필요한 패키지 설치
!pip install -q streamlit gymnasium gymnasium[classic-control] pygame
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install -q pyvirtualdisplay
print("설치 완료")

## 1. Gymnasium 설치 및 환경 생성

In [ ]:
import gymnasium as gym

# CartPole 환경 생성: 막대를 세운 채로 유지하는 게임
env = gym.make("CartPole-v1")

print("환경 생성 완료:", env.spec.id)

## 2. 환경 초기화 (reset)

In [ ]:
# 환경을 초기 상태로 리셋
state, info = env.reset()

labels = ["카트 위치", "카트 속도", "막대 각도", "막대 각속도"]
print("초기 상태 (observation):")
for i, (val, label) in enumerate(zip(state, labels)):
    print(f"  [{i}] {label:10s}: {val:+.6f}")
print("info:", info)

## 3. 한 스텝 실행 (step)

In [ ]:
# 행동: 0=왼쪽으로 밀기, 1=오른쪽으로 밀기
action = 1  # 오른쪽으로 밀기

next_state, reward, terminated, truncated, info = env.step(action)

labels = ["카트 위치", "카트 속도", "막대 각도", "막대 각속도"]
print("다음 상태 (observation):")
for i, (val, label) in enumerate(zip(next_state, labels)):
    print(f"  [{i}] {label:10s}: {val:+.6f}")
print("보상:", reward)
print("에피소드 종료(terminated):", terminated)
print("시간 초과(truncated):", truncated)

## 4. 랜덤 에이전트로 한 에피소드 실행

In [ ]:
import time

state, info = env.reset()
total_reward = 0
step_count = 0
labels = ["카트 위치", "카트 속도", "막대 각도", "막대 각속도"]

done = False
while not done:
    action = env.action_space.sample()  # 랜덤 행동
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
    step_count += 1

    # 스텝마다 출력
    action_str = "왼쪽" if action == 0 else "오른쪽"
    print(f"[스텝 {step_count}] 행동: {action_str} ({action})")
    for i, (val, label) in enumerate(zip(state, labels)):
        print(f"  [{i}] {label:10s}: {val:+.6f}")
    print(f"  보상: {reward}, 종료: {done}")
    print("-" * 40)

    time.sleep(0.4)  # 사람이 볼 수 있는 속도 (초)

print(f"에피소드 종료: {step_count} 스텝, 총 보상 {total_reward}")

env.close()

## 5. Streamlit 인터랙티브 데모

버튼으로 직접 CartPole을 조작해보세요. 아래 셀들을 순서대로 실행하면 `app.py`가 생성되고 공개 URL이 제공됩니다.

- **localtunnel** 접속 시 비밀번호로 본인 IP가 필요합니다. Sec.5b에서 출력된 IP를 입력하세요.

### 5a. app.py 생성 (블록 내에서 생성)

In [ ]:
app_py_code = '''import streamlit as st
import gymnasium as gym

st.set_page_config(layout="wide")
st.title("🛺 CartPole-v1 Gymnasium 환경 Streamlit 앱")

if "env" not in st.session_state:
    st.session_state.env = gym.make("CartPole-v1", render_mode="rgb_array")

def reset_environment():
    observation, info = st.session_state.env.reset()
    st.session_state.state = observation
    st.session_state.reward = 0.0
    st.session_state.terminated = False
    st.session_state.truncated = False
    st.session_state.total_reward = 0.0
    st.session_state.step_count = 0

def take_step(action):
    if st.session_state.terminated or st.session_state.truncated:
        st.warning("⚠️ 게임이 종료되었습니다. '다시 시작' 버튼을 눌러주세요.")
        return
    next_state, reward, terminated, truncated, info = st.session_state.env.step(action)
    st.session_state.state = next_state
    st.session_state.reward = reward
    st.session_state.terminated = terminated
    st.session_state.truncated = truncated
    st.session_state.total_reward += reward
    st.session_state.step_count += 1

if "state" not in st.session_state:
    reset_environment()

col1, col2 = st.columns([2, 1])
with col1:
    st.header("CartPole 시뮬레이션")
    img_array = st.session_state.env.render()
    st.image(img_array, caption=f"스텝: {st.session_state.step_count}, 총 보상: {st.session_state.total_reward:.1f}", width="stretch")

with col2:
    st.header("환경 정보 및 제어")
    st.subheader("현재 스텝 결과")
    st.write(f"**보상 (Reward):** {st.session_state.reward:.1f}")
    st.write(f"**에피소드 종료 (Terminated):** {'예' if st.session_state.terminated else '아니오'}")
    st.write(f"**시간 초과 (Truncated):** {'예' if st.session_state.truncated else '아니오'}")
    st.subheader("현재 상태 (Observation)")
    state_labels = ["카트 위치 (Cart Position)", "카트 속도 (Cart Velocity)", "막대 각도 (Pole Angle)", "막대 각속도 (Pole Angular Velocity)"]
    for i, (val, label) in enumerate(zip(st.session_state.state, state_labels)):
        st.write(f"  - **[{i}] {label}:** {val:+.6f}")
    st.subheader("행동 선택 (Action)")
    b1, b2 = st.columns(2)
    with b1:
        st.button("⬅️ 왼쪽으로 밀기", on_click=take_step, args=(0,))
    with b2:
        st.button("➡️ 오른쪽으로 밀기", on_click=take_step, args=(1,))
    if st.session_state.terminated or st.session_state.truncated:
        st.markdown("---")
        st.error("🎮 게임 오버!")
        st.write(f"최종 스텝 수: {st.session_state.step_count}")
        st.write(f"총 보상: {st.session_state.total_reward:.1f}")
        st.button("다시 시작", on_click=reset_environment)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_py_code)

print("app.py 생성 완료")

### 5b. Streamlit 실행 + localtunnel 공개 URL

아래 셀 실행 시 Streamlit이 백그라운드에서 실행되고 공개 URL이 생성됩니다. URL 클릭 후 5b에서 출력된 IP를 비밀번호로 입력하세요.

In [ ]:
import subprocess
import time

# 1. 프록시 비밀번호(IP) 출력 - localtunnel 접속 시 이 IP 입력
print("📌 프록시 비밀번호(IP):")
!curl -s ipv4.icanhazip.com
print()
# 2. Streamlit 백그라운드 실행
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(5)
# 3. localtunnel 공개 URL (이 셀은 실행 중 상태로 유지됨)
print("아래 URL을 브라우저에서 열고, 위 IP를 비밀번호로 입력하세요.\n")
!npx --yes localtunnel --port 8501